# MirrorTopology Step 1 — Phase D-3c：実 bank による 12 位置 profile・official gate・5 seed plan の固定（family 単位；notebook v0.3）
受入れ済みの固定入力（D-2 の第 1 波 bank＋family reference，D-3b の追加 bank（配列検証付き受入れ））を **formal intake** で読み（bytes／配列 SHA の再検証・ledger／receipt 束縛），1 family の 3 size × 12 位置 × 2 system（36 配置 × 2）の供給を組み立て，family 共有の 5 seed BootstrapPlan／FittingPlan を**実 cluster UID から 1 回だけ**構築（登録定数 B＝2000・B_KDE＝2000・seeds 5），size input 6 → 全 size 組立て → input fingerprint → **formal 12 位置 official gate（live 環境）** → gate 後の identity 再確認（内容 fingerprint＋plan object の同一性）→ profile record と plan identity の検証付き publication。新しい共分散・MC bank の生成，閾値／label／較正，D-2W，noise は行わない。入力は読取り専用。**High-RAM CPU runtime**（保持 payload の算術 ≈13.2 GB；plan 再構築照合の間は一時的に ≈16.5 GB，照合後に解放；51 GB が十分であることは保証されない：段階別 RSS／peak RSS を記録する）。既定で Drive の run を `/content/stage` に 1 回複製してから読む（intake は同じ NPZ を複数回開くため；複製前に disk 容量を検査；複製後も bytes／配列 SHA の再検証は同じ）。**family ごとに 1 run**（E2 → E7 → E8）。

**v0.3（監査 R-D3C1-A/launch-boundary・B/output-anchor）**：attempt cell は最初に **lock の出力先（anchor）** を検証し（symlink／非 directory／入力・source・stage root との交差／lock file の SHA 不一致なら何も書かずに停止），record はすべて anchor にだけ書く（現在の `OUT` が変更されていても使わない）。設定・lock file・live source の検査は 1 つの検査関数にまとめ，**staging の前**と**子 process 起動の直前**の 2 回呼ぶ（複製中に HEAD／tree／file SHA／lock／設定が変われば起動せず FAIL）。plan document の `source_lock.pins_sha256` も lock に照合する。

**v0.2（監査 R-D3C1-A／B）**：cell 2 は 1 回の **attempt** を表す。attempt ID を発行し，危険な処理（staging の削除／複製・子 process 起動）の前に「未完了・非成功」の initial record を `profile_final_record.json` に書く（前回の final record は `superseded_final_record_before_<attempt>.json` に退避し，今回の証拠として再利用しない）。その後，実行設定（FAMILY・入力 root・STAGE_LOCAL・path）と **live source**（実 HEAD・clean tree・inventory／script／ledger／receipt／pins の file SHA・engine 版）を cell 1 で固定した lock と再照合し，不一致なら子 process を呼ばず FAIL record を書く。staging（容量検査→複製）・起動・record の semantic 検証（型・family・attempt／lock／source の束縛・plan identity file の publication）・finalization の例外はすべて FAIL record に落ちる。`profile_pass` は rc 0 ∧ `D3C_PASS` ∧ gate passed ∧ record の shape 正常 ∧ 束縛一致 ∧ plan identity file 一致 のときだけ True。


In [1]:
# --- 0. OUTER LOCK (the only editable cell)
REPO_URL = 'https://github.com/tsujikeita/mirror-topology.git'
REPO_COMMIT = '7a2b17749e88074834bf5bebe3ebd8ac8af5dd29'
EXPECTED_INVENTORY_SHA256 = 'c0b0333b2f4250f1498c04b30090a1449d8135983264052bf602941852e580a9'
STAGE_LOCAL = True
STAGE_ROOT = '/content/stage'
FAMILY = 'E8'
D2_RUN_ROOT = '/content/drive/MyDrive/MirrorTopology_D2/E8_20260923T174910Z/out/d2'
D3B_RUN_ROOTS = {'L1.00': '/content/drive/MyDrive/MirrorTopology_D3b/E8_L1.00_20260930T213708Z/out/d3b',
                 'L1.20': '/content/drive/MyDrive/MirrorTopology_D3b/E8_L1.20_20261001T004059Z/out/d3b',
                 'L1.50': '/content/drive/MyDrive/MirrorTopology_D3b/E8_L1.50_20261001T132201Z/out/d3b'}

In [2]:
# --- 1. fresh scratch checkout; inventory + script + ledger + receipt bound; inputs exist and match the ledgers; Drive mounted (read-only usage); the LOCK (published, hashed)
import subprocess, sys, os, json, hashlib, shutil, time, re
sha=lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
assert re.fullmatch(r'[0-9a-f]{40}', REPO_COMMIT) and re.fullmatch(r'[0-9a-f]{64}', EXPECTED_INVENTORY_SHA256) and FAMILY in ('E2','E7','E8') and set(D3B_RUN_ROOTS)=={'L1.00','L1.20','L1.50'}
from google.colab import drive; drive.mount('/content/drive')
assert os.path.isfile(f'{D2_RUN_ROOT}/d2_bank_registry.json'), D2_RUN_ROOT
for s,p in D3B_RUN_ROOTS.items(): assert os.path.isfile(f'{p}/d3_bank_registry.json'), p
SCRATCH='/content/d3c_scratch'; shutil.rmtree(SCRATCH, ignore_errors=True); subprocess.run(['git','clone','-q',REPO_URL,SCRATCH],check=True); subprocess.run(['git','-C',SCRATCH,'checkout','-q',REPO_COMMIT],check=True)
assert subprocess.check_output(['git','-C',SCRATCH,'rev-parse','HEAD']).decode().strip()==REPO_COMMIT and subprocess.check_output(['git','-C',SCRATCH,'status','--porcelain']).decode().strip()==''
MT=SCRATCH; PHASEB=f'{SCRATCH}/engine/phaseB'; PHASEC=f'{SCRATCH}/phaseC'; INV=f'{PHASEB}/B2_completion_inventory.json'; assert sha(INV)==EXPECTED_INVENTORY_SHA256; inv=json.load(open(INV)); SCRIPT=f'{PHASEB}/d/d3c_profile.py'; assert sha(SCRIPT)==inv['d_sha256']['d/d3c_profile.py']
LEDGER=f'{PHASEB}/registered_assets/d3b/d3b_generation_ledger.json'; RECEIPT=f'{PHASEB}/registered_assets/d3b/d3b_outer_receipt.json'; PINS=f'{PHASEB}/d/d3_pins.json'; assert sha(LEDGER)==inv['registered_assets_sha256']['registered_assets/d3b/d3b_generation_ledger.json'] and sha(RECEIPT)==inv['registered_assets_sha256']['registered_assets/d3b/d3b_outer_receipt.json'] and sha(PINS)==inv['d_sha256']['d/d3_pins.json']
ledger=json.load(open(LEDGER)); d2l=json.load(open(f'{PHASEB}/registered_assets/d2/d2_generation_ledger.json'))
assert D2_RUN_ROOT.rstrip('/')==d2l['families'][FAMILY]['run_root'].rstrip('/')+'/out/d2', (D2_RUN_ROOT, d2l['families'][FAMILY]['run_root'])
for s,p in D3B_RUN_ROOTS.items(): assert p.rstrip('/')==ledger['partitions'][f'{FAMILY}_{s}']['run_root'].rstrip('/')+'/d3b', (s, p)
pins=json.load(open(PINS)); ex=pins['environment']; subprocess.run([sys.executable,'-m','pip','install','-q',f"numpy=={ex['numpy']}",f"scipy=={ex['scipy']}",f"healpy=={ex['healpy']}",f"camb=={ex['camb']}",f"pot=={ex['pot']}",'threadpoolctl','psutil'],check=True)
os.environ['OPENBLAS_NUM_THREADS']='2'; os.environ['OMP_NUM_THREADS']='2'
import platform, importlib; live={k: importlib.import_module({'pot':'ot'}.get(k,k)).__version__ for k in ('numpy','scipy','healpy','camb','pot')}; live['python']=platform.python_version(); mism={k:(live[k],ex[k]) for k in live if live[k]!=ex[k]}; assert not mism, f'environment lock failed (launcher pre-check): {mism}'
import psutil; vm=psutil.virtual_memory(); RAM_INFO=dict(total_bytes=vm.total, available_bytes=vm.available); print('RAM total GB', round(vm.total/1e9,1), 'available GB', round(vm.available/1e9,1)); assert vm.total>30e9, 'High-RAM runtime required'
OUT=f'/content/d3c_profile_{FAMILY}'; os.makedirs(OUT, exist_ok=True)      # attempts are separated inside OUT by their attempt id (nothing is deleted here)
# LOCK-BUILD (the lines from here to the end of this cell are re-used verbatim by the contract test)
def _publish(path, doc):
    data=json.dumps(doc, indent=1, ensure_ascii=False, default=str).encode('utf-8'); tmp=path+'.tmp'; open(tmp,'wb').write(data); os.replace(tmp, path); assert open(path,'rb').read()==data, path; return hashlib.sha256(data).hexdigest()
lock=dict(schema='d3c_launcher_lock_v2', commit=REPO_COMMIT, inventory_sha256=sha(INV), script_sha256=sha(SCRIPT), inventory_script_sha256=inv['d_sha256']['d/d3c_profile.py'], ledger_file_sha256=sha(LEDGER), ledger_sha256=ledger['ledger_sha256'], receipt_file_sha256=sha(RECEIPT), receipt_sha256=json.load(open(RECEIPT))['receipt_sha256'], pins_sha256=sha(PINS), engine=inv['engine_version'],
          family=FAMILY, d2_run_root=D2_RUN_ROOT, d3b_run_roots=dict(D3B_RUN_ROOTS), stage_local=bool(STAGE_LOCAL), stage_root=STAGE_ROOT, mt=MT, phaseb=PHASEB, phasec=PHASEC, script=SCRIPT, out=OUT, locked_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()))
LOCK_PATH=f'{OUT}/profile_lock.json'; LOCK_SHA256=_publish(LOCK_PATH, lock); print('lock', LOCK_SHA256); print(lock)


Mounted at /content/drive
RAM total GB 54.8 available GB 52.7
lock 79fd19e25c53fc6b36cd346949ecfa2f59e522f8dcfb2d5c3d41e6da2c721904
{'schema': 'd3c_launcher_lock_v2', 'commit': '7a2b17749e88074834bf5bebe3ebd8ac8af5dd29', 'inventory_sha256': 'c0b0333b2f4250f1498c04b30090a1449d8135983264052bf602941852e580a9', 'script_sha256': '9f243b08a5aa8ba0cc3ded42343fb8d99717dc4bd50ebf26a7cc9a519221638d', 'inventory_script_sha256': '9f243b08a5aa8ba0cc3ded42343fb8d99717dc4bd50ebf26a7cc9a519221638d', 'ledger_file_sha256': 'bdaa8aeb9b3e591d73540c4edce482c3f59f6edcafa0f21f171332102c624837', 'ledger_sha256': 'a7d2c8c4375c37a24142c1fd2bd400b9559c220418ee7af9e41f430ba968b9da', 'receipt_file_sha256': 'ed0fbccba739f26a906da0948885d3b4205ca6c58d5aa8338a62c877b166dabb', 'receipt_sha256': '7c42bc112a5413360837bc4e1d28f7583713e111d12e2bd903faf3966f402c18', 'pins_sha256': '136d238f84896d3c838b9972622076ba095486bee1cbd91f19460a4440063a46', 'engine': '0.99.0', 'family': 'E8', 'd2_run_root': '/content/drive/MyDrive/M

In [3]:
# --- 2. ONE ATTEMPT (v0.3; R-D3C1-A/launch-boundary + R-D3C1-B/output-anchor): (0) the SAFE anchored output is taken from the LOCK and validated BEFORE anything is written;
#        (1) attempt id + initial NON-success record at the anchor; (2) ONE verifier (configuration, lock file, LIVE source) runs BEFORE staging; (3) staging (disk check first);
#        (4) the SAME verifier runs again IMMEDIATELY before subprocess.run (no child process on any mismatch); (5) record validation + binding; final record at the anchor only
t0=time.time(); _in=lambda p,q: os.path.realpath(p)==os.path.realpath(q) or os.path.realpath(p).startswith(os.path.realpath(q).rstrip('/')+'/')
# 0. output anchor: from the in-memory lock (NOT the current OUT); a real non-symlink directory holding the published lock file; disjoint from the locked AND current inputs, the source and the stage root
ANCHOR=lock['out']; _clash=[q for q in [lock['d2_run_root'], *lock['d3b_run_roots'].values(), lock['mt'], lock['stage_root'], D2_RUN_ROOT, *D3B_RUN_ROOTS.values(), MT] if _in(ANCHOR,q) or _in(q,ANCHOR)]
_anchor_ok=bool(not os.path.islink(ANCHOR) and os.path.isdir(ANCHOR) and not _clash and LOCK_PATH==f'{ANCHOR}/profile_lock.json')      # the lock file's CONTENT is verified by the verifier below (a FAIL record at this anchor)
if not _anchor_ok: print(f'ATTEMPT NOT STARTED: the locked output {ANCHOR!r} is not a safe anchored directory (symlink / not a directory / clashes {_clash} / lock path not inside it); nothing was written', file=sys.stderr); raise RuntimeError('unsafe output anchor; nothing written')
ATTEMPT=time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())+'_'+hashlib.sha256(f'{time.time_ns()}|{FAMILY}|{REPO_COMMIT}|{LOCK_SHA256}'.encode()).hexdigest()[:10]
FIN=f'{ANCHOR}/profile_final_record.json'; RUN=f'{ANCHOR}/run_{ATTEMPT}'; profile_pass=False; IN_D2=None; IN_D3B=None; rc=None; v=None
REC=dict(schema='d3c_launcher_final_record_v3', attempt_id=ATTEMPT, started_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), lock_sha256=LOCK_SHA256, lock=lock, output_anchor=ANCHOR, ram=globals().get('RAM_INFO'), stage='started', profile_pass=False, exit_code=None, D3C_PASS=None, gate_passed=None, launcher_fallback=False, bindings_ok=None, failures=[], exception=None, run_dir=RUN, staged_inputs=None, bindings={}, seconds_total=None, finished_utc=None)
if os.path.exists(FIN): prev=f'{ANCHOR}/superseded_final_record_before_{ATTEMPT}.json'; os.replace(FIN, prev); REC['superseded_previous_record']=prev      # a previous attempt's record is NEVER this attempt's evidence
_publish(FIN, REC)
def _finish(stage, failures=(), exc=None):
    REC.update(stage=stage, exception=(repr(exc) if exc is not None else None), seconds_total=round(time.time()-t0, 3), finished_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())); REC['failures']+=list(failures); REC['profile_pass']=bool(profile_pass); _publish(FIN, REC); print('attempt', ATTEMPT, '| stage', stage, '| profile_pass', REC['profile_pass'], '| failures', REC['failures'])
def _verify_locked_state(prefix):
    # the ONE verifier: current configuration == lock; lock file on disk == lock; LIVE source (actual HEAD, clean tree, file SHAs of inventory / script / ledger / receipt / pins,
    # the inventory's script SHA, engine version, tree layout) == lock; input roots present. Called before staging and again immediately before the child launch.
    REC['stage']=f'{prefix}_config'; cfg=dict(family=FAMILY, d2_run_root=D2_RUN_ROOT, d3b_run_roots=dict(D3B_RUN_ROOTS), stage_local=bool(STAGE_LOCAL), stage_root=STAGE_ROOT, commit=REPO_COMMIT, mt=MT, phaseb=PHASEB, phasec=PHASEC, script=SCRIPT, out=OUT)
    mism={k: (cfg[k], lock.get(k)) for k in cfg if cfg[k]!=lock.get(k)}
    if mism: raise RuntimeError(f'configuration differs from the lock (re-run the preflight cell): {mism}')
    if LOCK_PATH!=f'{ANCHOR}/profile_lock.json' or sha(LOCK_PATH)!=LOCK_SHA256 or json.load(open(LOCK_PATH))!=lock: raise RuntimeError('lock file on disk differs from the lock in memory')
    REC['stage']=f'{prefix}_source'; inv_now=json.load(open(INV))
    src=dict(head=subprocess.check_output(['git','-C',MT,'rev-parse','HEAD'],text=True).strip(), clean=(subprocess.check_output(['git','-C',MT,'status','--porcelain'],text=True).strip()==''), inventory_sha256=sha(INV), script_sha256=sha(SCRIPT), inventory_script_sha256=inv_now['d_sha256']['d/d3c_profile.py'], ledger_file_sha256=sha(LEDGER), receipt_file_sha256=sha(RECEIPT), pins_sha256=sha(PINS), engine=inv_now['engine_version'], checked_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()))
    want=dict(head=lock['commit'], clean=True, inventory_sha256=lock['inventory_sha256'], script_sha256=lock['script_sha256'], inventory_script_sha256=lock['script_sha256'], ledger_file_sha256=lock['ledger_file_sha256'], receipt_file_sha256=lock['receipt_file_sha256'], pins_sha256=lock['pins_sha256'], engine=lock['engine'])
    bad=[k for k in want if src[k]!=want[k]]; REC['bindings'][f'live_source_{prefix}']=src
    if bad or PHASEB!=f'{MT}/engine/phaseB' or PHASEC!=f'{MT}/phaseC' or SCRIPT!=f'{PHASEB}/d/d3c_profile.py': raise RuntimeError(f'live source differs from the lock ({prefix}): {bad}')
    missing=[p for p in [D2_RUN_ROOT, *D3B_RUN_ROOTS.values()] if not os.path.isdir(p)]
    if missing: raise RuntimeError(f'input roots missing: {missing}')
try:
    # 2. configuration / lock file / live source BEFORE staging (editing cell 0 after the preflight, or skipping the preflight, is refused here)
    _verify_locked_state('precheck'); _publish(FIN, REC)
    # 3. optional local staging: disk capacity is checked BEFORE the copy; the script re-verifies every byte / array SHA of the staged copy
    REC['stage']='staging'; IN_D2=D2_RUN_ROOT; IN_D3B=dict(D3B_RUN_ROOTS)
    if STAGE_LOCAL:
        ST=STAGE_ROOT; guard=[q for q in [D2_RUN_ROOT, *D3B_RUN_ROOTS.values(), ANCHOR, MT] if _in(ST,q) or _in(q,ST)]
        if guard: raise RuntimeError(f'STAGE_ROOT must be disjoint from the inputs, the output and the source: {guard}')
        need=sum(os.path.getsize(os.path.join(d,f)) for root in [D2_RUN_ROOT, *D3B_RUN_ROOTS.values()] for d,_,fs in os.walk(root) for f in fs); ts=time.time()
        shutil.rmtree(ST, ignore_errors=True); os.makedirs(ST); free=shutil.disk_usage(ST).free; REC['staging']=dict(root=ST, input_bytes=need, disk_free_bytes_before=free)
        if free < need*1.05+2e9: raise RuntimeError(f'insufficient local disk for staging: need {need/1e9:.1f} GB (+5% +2 GB margin), free {free/1e9:.1f} GB')
        IN_D2=f'{ST}/d2'; shutil.copytree(D2_RUN_ROOT, IN_D2); IN_D3B={s: f'{ST}/d3b_{s}' for s in D3B_RUN_ROOTS}
        for s,p in D3B_RUN_ROOTS.items(): shutil.copytree(p, IN_D3B[s])
        REC['staging'].update(seconds=round(time.time()-ts,1), disk_free_bytes_after=shutil.disk_usage(ST).free); print('staged in', round(time.time()-ts), 's; disk free GB', round(REC['staging']['disk_free_bytes_after']/1e9,1))
    REC['staged_inputs']=dict(d2=IN_D2, d3b=IN_D3B); _publish(FIN, REC)
    # 4. the SAME verifier IMMEDIATELY before the launch (state may have changed during the long copy): any mismatch -> FAIL record, NO child process
    _verify_locked_state('prelaunch')
    REC['stage']='launch'; args=[sys.executable,SCRIPT,'--mt',MT,'--phaseb',PHASEB,'--phasec',PHASEC,'--out',RUN,'--family',FAMILY,'--d2-root',IN_D2,'--profile','production_official','--attempt-id',ATTEMPT,'--launcher-lock-sha256',LOCK_SHA256]+[x for s in sorted(IN_D3B) for x in ('--d3b-root', f'{s}={IN_D3B[s]}')]
    REC['args']=args; _publish(FIN, REC); rc=subprocess.run(args,capture_output=True,text=True); REC['exit_code']=rc.returncode
    open(f'{ANCHOR}/profile_stdout_{ATTEMPT}.txt','w').write(rc.stdout); open(f'{ANCHOR}/profile_stderr_{ATTEMPT}.txt','w').write(rc.stderr); print(rc.stdout[-3000:]); print('rc', rc.returncode)
    # 5. semantic validation of the record (shape -> launcher_fallback) and its binding to THIS attempt / lock / source (-> bindings_ok); plan identity file published and bound (incl. pins)
    REC['stage']='record'; vp=f'{RUN}/d3c_profile_record_{FAMILY}.json'; shape=[]; binding=[]; g={}
    try: v=json.load(open(vp))
    except Exception as read_error: v=None; shape.append(f'profile record unreadable: {read_error!r}')
    if v is not None and not isinstance(v, dict): shape.append('profile record is not a dictionary'); v=None
    if v is not None:
        g=v.get('gate'); num=lambda x: isinstance(x,(int,float)) and not isinstance(x,bool)
        for k, ok in [('D3C_PASS', isinstance(v.get('D3C_PASS'), bool)), ('stage', isinstance(v.get('stage'), str)), ('gates', isinstance(v.get('gates'), dict)), ('failures', isinstance(v.get('failures'), list)), ('seconds', num(v.get('seconds'))), ('stages_rss_mb', isinstance(v.get('stages_rss_mb'), dict)),
                      ('gate', g is None or (isinstance(g, dict) and isinstance(g.get('passed'), bool) and isinstance(g.get('required_failures'), list))), ('attempt', isinstance(v.get('attempt'), dict)), ('selftest', v.get('selftest') is False), ('profile', v.get('profile')=='production_official')]:
            if not ok: shape.append(f'profile record field invalid: {k}')
        g=g if isinstance(g, dict) else {}; a_=v.get('attempt') if isinstance(v.get('attempt'), dict) else {}; s_=v.get('source') if isinstance(v.get('source'), dict) else {}
        for k, ok in [('family', v.get('family')==FAMILY==lock['family']), ('attempt_id', a_.get('attempt_id')==ATTEMPT), ('launcher_lock_sha256', a_.get('launcher_lock_sha256')==LOCK_SHA256), ('source.script_sha256', s_.get('script_sha256')==lock['script_sha256']), ('source.inventory_sha256', s_.get('inventory_sha256')==lock['inventory_sha256']), ('source.pins_sha256', s_.get('pins_sha256')==lock['pins_sha256']), ('source.engine_version', s_.get('engine_version')==lock['engine'])]:
            if not ok: binding.append(f'profile record not bound to this attempt / lock: {k}')
        pp=f'{RUN}/d3c_plan_identity_{FAMILY}.json'; pe=(v.get('published_evidence') or {}).get(f'd3c_plan_identity_{FAMILY}.json') if isinstance(v.get('published_evidence'), dict) else None
        if os.path.isfile(pp):
            pd=json.load(open(pp)); pa=pd.get('attempt') if isinstance(pd, dict) and isinstance(pd.get('attempt'), dict) else {}; ps=pd.get('source_lock') if isinstance(pd, dict) and isinstance(pd.get('source_lock'), dict) else {}
            plan_ok=bool(isinstance(pe, dict) and pe.get('sha256')==sha(pp) and pe.get('bytes')==os.path.getsize(pp) and isinstance(pd, dict) and pd.get('family')==FAMILY and pd.get('formal') is True and pd.get('selftest') is False and pa.get('attempt_id')==ATTEMPT and pa.get('launcher_lock_sha256')==LOCK_SHA256 and ps.get('script_sha256')==lock['script_sha256'] and ps.get('inventory_sha256')==lock['inventory_sha256'] and ps.get('pins_sha256')==lock['pins_sha256'] and (pd.get('identity') or {}).get('identity_sha256')==v.get('plan_identity_sha256'))
            if not plan_ok: binding.append('plan identity file not published / not bound to the record, this attempt and the lock (incl. pins)')
        else:
            plan_ok=False
            if v.get('D3C_PASS') is True: binding.append('plan identity file missing: the record claims D3C_PASS but is not bound to a published plan identity')
        REC.update(D3C_PASS=v.get('D3C_PASS'), gate_passed=g.get('passed'), record_sha256=sha(vp), record_stage=v.get('stage'), record_failures=v.get('failures'), gates=v.get('gates'), stages_rss_mb=v.get('stages_rss_mb'), stages_peak_rss_mb=v.get('stages_peak_rss_mb'), seconds_script=v.get('seconds'), plan_identity_sha256=v.get('plan_identity_sha256'), plan_identity_file_ok=plan_ok)
        print('rc', rc.returncode, '| stage', v.get('stage'), '| D3C_PASS', v.get('D3C_PASS'), '| gate passed', g.get('passed'), '| failures', v.get('failures'), '| rss MB', v.get('stages_rss_mb'), '| peak rss MB', v.get('stages_peak_rss_mb'), '| seconds', v.get('seconds'))
    else: plan_ok=False
    REC.update(launcher_fallback=bool(shape), bindings_ok=(not binding) if v is not None else None, stderr_tail=rc.stderr[-2000:])
    profile_pass=bool(rc.returncode==0 and v is not None and v.get('D3C_PASS') is True and g.get('passed') is True and not shape and not binding and plan_ok)
    _finish('complete', shape+binding)
except Exception as attempt_error:
    profile_pass=False; _finish(REC.get('stage') or 'unknown', [f'exception at stage {REC.get("stage")}: {attempt_error!r}'], attempt_error)
print('profile_pass', profile_pass)


staged in 655 s; disk free GB 195.1
  intake 40101 (L1.00 position 0, reuse_D2_fixed_input) 6s rss 1587 MB
  intake 40102 (L1.00 position 1, reuse_D2_fixed_input) 5s rss 2035 MB
  intake 40103 (L1.00 position 2, reuse_D2_fixed_input) 5s rss 2342 MB
  intake 40104 (L1.00 position 3, generate_D3b) 7s rss 2615 MB
  intake 40105 (L1.00 position 4, generate_D3b) 7s rss 2911 MB
  intake 40106 (L1.00 position 5, generate_D3b) 7s rss 3218 MB
  intake 40107 (L1.00 position 6, generate_D3b) 7s rss 3502 MB
  intake 40108 (L1.00 position 7, generate_D3b) 7s rss 3798 MB
  intake 40109 (L1.00 position 8, generate_D3b) 7s rss 4081 MB
  intake 40110 (L1.00 position 9, generate_D3b) 7s rss 4377 MB
  intake 40111 (L1.00 position 10, generate_D3b) 7s rss 4648 MB
  intake 40112 (L1.00 position 11, generate_D3b) 7s rss 4956 MB
  intake 40201 (L1.20 position 0, reuse_D2_fixed_input) 6s rss 5251 MB
  intake 40202 (L1.20 position 1, reuse_D2_fixed_input) 5s rss 5510 MB
  intake 40203 (L1.20 position 2, reuse_

In [4]:
# --- 3. zip the evidence (small: lock, attempt records (current + superseded), run records, plan identity, stdout/stderr) for the audit
zp=f'/content/d3c_profile_{FAMILY}_{REPO_COMMIT[:12]}.zip'; shutil.make_archive(zp[:-4], 'zip', root_dir=lock['out'])
from google.colab import files; print(zp, os.path.getsize(zp)); files.download(zp)

/content/d3c_profile_E8_7a2b17749e88.zip 128226


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>